# Create Multi Agents to Research and Write an A
* ***AI-Related Blog Article or Research Paper Using CrewAI***

# install lib

In [63]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [64]:
%pip install "crewai[tools]"
%pip install -U google-genai


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [65]:
import google.genai

print("Google GenAI SDK installed successfully")

Google GenAI SDK installed successfully


# Setup

In [ ]:
import os
import asyncio
from crewai import Agent, LLM, Task, Crew, Process
from dotenv import load_dotenv

load_dotenv(override=True)
print("Environment variables loaded.")

gemini_api_key = os.getenv("GEMINI_API_KEY")

llm = LLM(
    model="gemini/gemini-3.6-flash",  # Must include 'gemini/' prefix
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=1.0
)

Environment variables loaded.


# Creating Agent

In [67]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True,
    llm=llm,
    max_tokens=1500
)

In [68]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True,
    llm = llm,
    max_tokens=2000
)

In [69]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True,
    llm = llm,
    max_tokens=1500
)

# Creating Tasks

***Task: Plan***

In [70]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

***Task: Write***

In [71]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

***Task: Edit***

In [72]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

# Creating the Crew

In [73]:
multiAgentCrew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)


# Running the Crew

In [74]:
import asyncio

async def run_crew():
    result = await multiAgentCrew.akickoff(inputs={'topic': 'Agentic AI'})
    return result

# In Jupyter, top-level await works directly
result = await run_crew()
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3aa3df1b-c020-4643-96af-0d1eee6efac3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Agentic AI.                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: c2e85b3d-08ac-4610-a450-1a8bec91b844                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Agentic AI.                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Google Gemini API error: 400 - API key not valid. Please pass a valid API key.


An unknown error occurred. Please check the details below.
Error details: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Agentic AI.                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 3aa3df1b-c020-4643-96af-0d1eee6efac3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}

In [ ]:
from IPython.display import Markdown
Markdown(result.raw)

# Artificial Intelligence Trends 2025: How to Drive Enterprise Value in the Next Era of AI

## Introduction

The enterprise perspective on artificial intelligence has fundamentally shifted. The period between 2022 and 2023 was defined by widespread public awe at conversational interfaces, but current **Artificial Intelligence trends 2025** reveal a far more pragmatic landscape. Organizations are moving past simple chat interfaces and novelty applications, shifting their focus toward deep infrastructure integration, agentic workflows, and measurable enterprise value. The conversational prompt box is no longer the endgame; it is merely the user interface of an increasingly complex machine learning ecosystem.

This operational shift is backed by hard enterprise data. According to the **McKinsey Global Survey on AI (2024)**, 72% of organizations have adopted AI in at least one business function, up significantly from 55% in 2023. Technology leaders are no longer asking whether artificial intelligence works; they are asking how to deploy it safely, sustainably, and profitably. Modern operational success demands moving beyond generic tool sets toward tailored **AI business integration strategies**, rigorous data governance, and specialized architectural paradigms like Agentic AI and Multimodal models.

Consequently, the era of passive experimentation with artificial intelligence is yielding to an era of structural operational integration. Technology executives who treat AI as a mere add-on feature risk falling behind competitors who view it as foundational digital infrastructure. Navigating this evolving landscape requires examining the macro forces reshaping the technology market, analyzing core enterprise adoption trends, evaluating critical implementation hurdles, and establishing a structured framework to secure tangible operational return on investment.

---

## The Current AI Landscape: Key Players and Paradigm Shifts

The artificial intelligence ecosystem has matured into a multi-layered market driven by distinct competitive priorities. At the top of the stack sit frontier model developers—such as OpenAI, Anthropic (with its Claude 3.5 suite), and Google DeepMind—who continue to push the boundaries of foundational model capability. Concurrently, the open-source movement, spearheaded by Meta’s Llama series, has democratized access to high-performing Large Language Models (LLMs). Enterprise integrators such as Microsoft, AWS, Salesforce, and IBM are embedding these foundational capabilities directly into software applications, bridging the gap between baseline research and operational utility.

A central strategic decision facing tech decision-makers is navigating the trade-off between proprietary API services and open-source models. Proprietary solutions often offer state-of-the-art capability out of the box with minimal infrastructure overhead, but they come with vendor lock-in, recurring token expenditures, and potential data sovereignty concerns. Conversely, open-source models allow enterprises to retain full ownership of model weights and execution environments, offering superior long-term cost controls and data privacy, though they require internal machine learning talent to deploy and maintain.

| Criteria | Proprietary Models (e.g., OpenAI, Anthropic) | Open-Source Models (e.g., Meta Llama, Mistral) |
| :--- | :--- | :--- |
| **Upfront Cost & Overhead** | Low (Pay-per-token API model) | Moderate to High (Requires compute infrastructure) |
| **Data Privacy & Control** | Dependent on vendor terms & private cloud instances | Complete control over model weights and enterprise data |
| **Customizability** | Limited to fine-tuning and prompting parameters | Deep architectural modifications and targeted fine-tuning |
| **Operational Latency** | Governed by third-party cloud infrastructure | Variable; optimized based on localized deployment |

Strategic consensus suggests that relying exclusively on generic, third-party proprietary APIs without internal data architectures can limit an organization's long-term competitive advantage. While proprietary models provide an exceptional starting point for rapid prototyping, enduring enterprise value stems from owning data pipelines, contextual domain knowledge, and targeted orchestration layers that sit around core intelligence engines.

---

## Core Trends Shaping the Future of Business AI

### Trend 1: The Emergence of Agentic AI
The transition from passive text generation to **agentic AI applications** represents a fundamental advancement in software design. Unlike traditional chatbots that require continuous human prompts, autonomous AI agents can reason through complex objectives, decompose tasks into logical steps, utilize external APIs, and execute end-to-end workflows independently. **Gartner** forecasts that by 2028, at least 15% of day-to-day work decisions will be made autonomously through agentic AI, up from virtually 0% in 2024.

This paradigm shift is already transforming domain-specific operations across multiple sectors. From automated supply chain re-routing and software engineering debugging to multi-tier financial auditing, agentic systems reduce manual bottlenecks. By connecting task planning with execution tools, organizations can automate complex operations that previously required extensive human intervention.

### Trend 2: Multimodal Systems Become the Baseline
Standard text-only interfaces are rapidly giving way to unified **multimodal AI models**. Leading platforms can natively process text, visual data, audio streams, and real-time video feeds concurrently within a unified model context. This evolution dramatically expands the enterprise surface area for automation across operational functions.

In field service and manufacturing, for example, an inspector can stream real-time video of industrial equipment while an embedded multimodal model diagnoses mechanical wear. By cross-referencing technical manuals via Natural Language Processing (NLP), the system can instantly draft a work order, significantly shortening resolution cycles and improving asset reliability.

### Trend 3: Small Language Models (SLMs) and Infrastructure Realities
As computational scale demands grow, reliance on massive centralized models is encountering financial and practical limitations. The **Stanford AI Index Report (2024)** highlights that training costs for state-of-the-art frontier models have escalated exponentially, making architectural efficiency a top corporate mandate. As a result, Small Language Models (SLMs) paired with Edge Computing are gaining momentum across enterprise environments.

SLMs deliver targeted domain performance at a fraction of the parameter size, enabling faster execution times, lower power consumption, and localized data processing directly on enterprise devices. Rather than deploying massive 100-billion-parameter models for localized tasks, forward-thinking technical leaders prioritize targeted SLMs for routine operations, reserving high-cost frontier models for tasks that demand complex, broad-reasoning capabilities.

```
[ Evolution of Enterprise AI Capabilities ]

+-------------------------------+
|  Level 1: Basic Automation    | -> Rule-based scripts & static RPA
+-------------------------------+
               |
+-------------------------------+
|  Level 2: Conversational AI   | -> Passive LLM chatbots & text tools
+-------------------------------+
               |
+-------------------------------+
|  Level 3: Multimodal Systems  | -> Text, audio, image & video integration
+-------------------------------+
               |
+-------------------------------+
|  Level 4: Autonomous Agents   | -> Multi-step planning & independent action
+-------------------------------+
```

---

## Overcoming Enterprise Implementation Bottlenecks

Despite accelerated interest, **enterprise AI implementation** continues to encounter significant operational friction. The primary hurdle for legacy organizations is data readiness, as foundational models are only as effective as the underlying data environments that feed them. Rather than undertaking continuous, costly model fine-tuning, leading organizations are deploying Retrieval-Augmented Generation (RAG) architectures. RAG bridges static model parameter memory with dynamic internal enterprise databases, allowing contextual, real-time data retrieval while maintaining strict enterprise Data Governance controls.

Security, intellectual property protection, and regulatory compliance represent another critical layer of complexity. With global enforcement mechanisms like the **EU AI Act**, enterprises face strict legal parameters governing risk tiering, model transparency, and data lineage. Maintaining an **EU AI Act compliance checklist for businesses** is now a fundamental requirement for global operations. Highlighting this shift, **IDC** reports that 60% of G2000 companies will spend more on AI compliance and governance mechanisms than on specialized AI security tooling by late 2025.

```
[ Enterprise Data Pipeline: RAG Architecture ]

+-------------------------+     +-----------------------+     +------------------------+
| Enterprise Vector Data  | --> | Contextual Retrieval  | --> | Secure Prompt Layer    |
| (Private Databases)     |     | (RAG Engine)          |     | (Data Masking/Filter)  |
+-------------------------+     +-----------------------+     +------------------------+
                                                                          |
                                                                          v
+-------------------------+                                   +------------------------+
| Verified Output /       | <-------------------------------- | Model Processing       |
| Actionable Workflow     |                                   | (SLM or Frontier API)  |
+-------------------------+                                   +------------------------+
```

Crucially, technological limitations are rarely the sole point of failure for corporate technology initiatives; organizational inertia and incomplete risk frameworks pose equal challenges. Building an effective enterprise AI strategy requires upskilling internal talent, establishing cross-functional AI Centers of Excellence (CoE), and maintaining clear human-in-the-loop operational guardrails to audit automated decisions and mitigate operational risk.

---

## Practical Strategic Framework: Moving from Hype to High ROI

To establish a clear path toward positive financial impact, decision-makers must implement structured deployment frameworks. Strategic focus should prioritize high-frequency, labor-intensive business workflows that possess mature digital data pipelines. Determining **how to implement generative AI in enterprise workflows** requires evaluating potential initiatives based on process repeatability, risk profiles, and direct connection to core bottom-line operations.

```
[ Use Case Evaluation Matrix ]

High Impact |  (1) Strategic Focus  |  (2) Long-Term R&D  |
            |   (Agentic Workflows) |   (Custom Frontier) |
            +-----------------------+---------------------+
Low Impact  |  (3) Quick Wins       |  (4) Deprioritize   |
            |   (Standard Copilots) |   (Niche Experiments)|
            +-----------------------+---------------------+
                       Low Complexity         High Complexity
```

A disciplined deployment approach balances off-the-shelf software purchases against custom engineering initiatives. Standard productivity tasks benefit from integrated enterprise copilot assistants, whereas core operational workflows demand customized RAG pipelines and tailored agent architectures. Regardless of the chosen path, organizations must implement real-time execution monitoring to prevent hallucinations, track API usage, and accurately measure efficiency gains.

Measuring the **ROI of generative AI in business** requires looking beyond generic productivity metrics in favor of explicit operational tracking, such as workflow velocity, task completion rates, and error reduction metrics. **PwC** projects that artificial intelligence could contribute up to **$15.7 trillion** to the global economy by 2030, driven predominantly by enterprise productivity enhancements. Establishing baseline metrics prior to deployment ensures organizations can track and validate these financial returns effectively.

---

## Conclusion: Navigating the Next Era of Artificial Intelligence

As **Artificial Intelligence trends 2025** take shape, the technology is transitioning from an experimental novelty into core corporate infrastructure. Organizations that successfully navigate this shift will move beyond simple conversational tools, deploying agentic workflows, multimodal capabilities, dynamic RAG architectures, and efficient Small Language Models to build durable operational advantages.

Market leadership in this next era will belong not to organizations that deploy breakthroughs the fastest, but to those that integrate them most responsibly within robust governance frameworks, secure data infrastructure, and clear corporate strategy. Balancing rapid adoption with thoughtful execution remains the definitive blueprint for sustainable competitive advantage.

Ready to transition your organization from passive AI experimentation to measurable business value? **Download our Enterprise AI Readiness Assessment Checklist** to evaluate your internal infrastructure, or **subscribe to our Tech Strategy Newsletter** for weekly executive insights. What is your organization’s primary hurdle in deploying enterprise AI this year? Share your perspectives and join the conversation in the comments below.